# Лабораторная работа 12. Квантизация и speculative decoding (студенческая версия)

**Курс:** Методы обработки естественного языка · **Блок 4:** инференс и эффективность

**Цель.** Реализовать два основных способа ускорить инференс без переобучения модели и проверить их теоретические свойства.

**Что нужно сдать:**
1. Absmax- и zero-point-квантизацию int8/int4 с разной гранулярностью и таблицу perplexity.
2. Построение NF4 и сравнение ошибки квантизации с int4 на реальных весах.
3. Анализ выбросов в активациях и их влияния на квантизацию активаций.
4. Упрощённый GPTQ для одного линейного слоя и сравнение с round-to-nearest.
5. Speculative sampling: эмпирическую проверку совпадения распределений (критерий хи-квадрат) и измерение ускорения.

**Теория.**
- **Absmax (симметричная):** \(s = \max|w| / (2^{b-1}-1)\), \(\hat w = s\cdot\mathrm{round}(w/s)\).
- **Zero-point (асимметричная):** \(s = (\max w - \min w)/(2^b - 1)\), \(z = \mathrm{round}(-\min w / s)\), \(\hat w = s(\mathrm{clamp}(\mathrm{round}(w/s) + z) - z)\).
- **GPTQ** минимизирует \(\|XW^\top - X\hat W^\top\|^2\) по столбцам: после квантизации столбца \(j\) ошибка \(\delta_j = (w_j - \hat w_j)/[H^{-1}]_{jj}\) компенсируется обновлением ещё не квантованных столбцов \(w_{j+1:} \mathrel{-}= \delta_j [H^{-1}]_{j,j+1:}\), где \(H = 2X^\top X\) (Frantar et al., 2023; OBS).
- **Speculative sampling:** draft-модель \(q\) предлагает токен \(x\), целевая модель \(p\) принимает его с вероятностью \(\min(1, p(x)/q(x))\); при отказе токен берётся из \(\mathrm{norm}(\max(0, p - q))\). Итоговое распределение в точности равно \(p\).

In [ ]:
# ==== Общая настройка (одинакова во всех работах) ====
import os, math, time, json, random, pathlib, urllib.request
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

FAST_DEV_RUN = True          # True — сокращённый прогон (CPU, минуты); False — полный эксперимент
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ARTIFACTS = pathlib.Path("artifacts"); ARTIFACTS.mkdir(exist_ok=True)
DATA = pathlib.Path("data"); DATA.mkdir(exist_ok=True)

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed()

def download(url, name):
    path = DATA / name
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    return path

def load_shakespeare():
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    return download(url, "shakespeare.txt").read_text(encoding="utf-8")

def load_russian():
    """Русский корпус: «Война и мир» (Л. Н. Толстой), т. 1, из открытого репозитория.
    При недоступности сети используется встроенный фрагмент."""
    urls = ["https://raw.githubusercontent.com/mmcdermott/nlp-war-and-peace/master/data/war_and_peace_ru.txt",
            "https://raw.githubusercontent.com/nevmenandr/word2vec-russian-novels/master/books_before/WarAndPeace.txt"]
    for i, u in enumerate(urls):
        try:
            t = download(u, f"ru_{i}.txt").read_text(encoding="utf-8", errors="ignore")
            if len(t) > 100_000: return t
        except Exception:
            pass
    base = ("Все счастливые семьи похожи друг на друга, каждая несчастливая семья несчастлива по-своему. "
            "Всё смешалось в доме Облонских. Жена узнала, что муж был в связи с бывшею в их доме француженкою-гувернанткой, "
            "и объявила мужу, что не может жить с ним в одном доме. ")
    return base * 400

print(f"torch {torch.__version__}, device={DEVICE}, FAST_DEV_RUN={FAST_DEV_RUN}")

In [ ]:
# ==== Загрузка предобученных моделей Hugging Face ====
from transformers import AutoTokenizer, AutoModelForCausalLM

def load_lm(name, dtype=None):
    tok = AutoTokenizer.from_pretrained(name)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    dtype = dtype or (torch.float16 if DEVICE == "cuda" else torch.float32)
    try:
        model = AutoModelForCausalLM.from_pretrained(name, dtype=dtype)
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=dtype)
    return model.to(DEVICE).eval(), tok

@torch.no_grad()
def perplexity(model, tok, text, max_len=512, stride=256, max_windows=None):
    ids = tok(text, return_tensors="pt").input_ids.to(DEVICE)
    nll, cnt, prev = 0.0, 0, 0
    for i, start in enumerate(range(0, ids.size(1) - 1, stride)):
        if max_windows and i >= max_windows: break
        end = min(start + max_len, ids.size(1)); x = ids[:, start:end]
        tgt = x.clone(); tgt[:, : max(0, prev - start)] = -100
        out = model(x, labels=tgt)
        n = (tgt[:, 1:] != -100).sum().item(); nll += out.loss.float().item() * n; cnt += n; prev = end
        if end == ids.size(1): break
    return math.exp(nll / cnt)

In [ ]:
TARGET, DRAFT = ("HuggingFaceTB/SmolLM2-360M", "HuggingFaceTB/SmolLM2-135M") if FAST_DEV_RUN else ("Qwen/Qwen2.5-1.5B", "Qwen/Qwen2.5-0.5B")
model, tok = load_lm(DRAFT, dtype=torch.float32)          # квантизация исследуется на меньшей модели
eval_text = load_shakespeare()[-(20_000 if FAST_DEV_RUN else 200_000):]
MAXW = 8 if FAST_DEV_RUN else None
base_ppl = perplexity(model, tok, eval_text, max_windows=MAXW)
print(f"{DRAFT}: perplexity FP32 = {base_ppl:.2f}")

## 1. Absmax и zero-point квантизация весов

Реализуйте fake-quantization (квантизация + деквантизация) с гранулярностью:
- `per_tensor` — один масштаб на матрицу;
- `per_channel` — масштаб на выходной канал (строку);
- `group` — масштаб на группу из `group_size` подряд идущих входных весов строки.

Квантуйте все линейные слои трансформера (кроме эмбеддингов и выходной головы) и измерьте perplexity.

In [ ]:
def quantize(w, bits=8, mode="absmax", granularity="per_channel", group_size=128):
    shape = w.shape
    if granularity == "per_tensor": x = w.reshape(1, -1)
    elif granularity == "per_channel": x = w.reshape(shape[0], -1)
    else: x = w.reshape(-1, group_size)
    # TODO: ваш код
    raise NotImplementedError
    return xq.reshape(shape)

import copy
def quantize_model(model, fn):
    m = copy.deepcopy(model)
    with torch.no_grad():
        for name, mod in m.named_modules():
            if isinstance(mod, nn.Linear) and "lm_head" not in name:
                mod.weight.copy_(fn(mod.weight.data))
    return m

w = torch.randn(256, 512)
for b in (8, 4):
    for g in ("per_tensor", "per_channel", "group"):
        print(f"int{b} {g:12s} rel. MSE = {((quantize(w, b, granularity=g) - w).pow(2).mean() / w.pow(2).mean()).item():.2e}")

In [ ]:
import pandas as pd
rows = [{"схема": "FP32", "perplexity": base_ppl}]
configs = [(8, "absmax", "per_tensor"), (8, "absmax", "per_channel"), (4, "absmax", "per_tensor"), (4, "absmax", "per_channel"),
           (4, "absmax", "group"), (4, "zeropoint", "per_channel"), (4, "zeropoint", "group"), (3, "absmax", "group")]
# TODO: ваш код (perplexity для каждой конфигурации)

## 2. NF4

NF4 (Dettmers et al., 2023) — 16 уровней, равных квантилям стандартного нормального распределения, нормированных на \([-1, 1]\), с точным нулём: 8 положительных уровней и 7 отрицательных плюс 0. Квантизация блочная: масштаб — absmax блока из 64 весов, каждое нормированное значение заменяется ближайшим уровнем.

Сравните относительную MSE для NF4 и int4 absmax (блок 64) на весах реальной модели и на синтетических данных: гауссовых и равномерных.

In [ ]:
from scipy.stats import norm

def nf4_levels(offset=0.9677):
    # TODO: ваш код
    raise NotImplementedError

NF4 = nf4_levels(); print("уровни NF4:", NF4.numpy().round(4)); assert len(NF4) == 16 and (NF4 == 0).any()

def quantize_nf4(w, block=64):
    # TODO: ваш код
    raise NotImplementedError

def rel_mse(a, b): return ((a - b).pow(2).mean() / b.pow(2).mean()).item()
W_real = torch.cat([m.weight.data.flatten()[: 2 ** 16] for n, m in model.named_modules() if isinstance(m, nn.Linear) and "lm_head" not in n][:20])
for name, wt in [("реальные веса", W_real), ("N(0,1)", torch.randn(2 ** 18)), ("U(-1,1)", torch.rand(2 ** 18) * 2 - 1)]:
    print(f"{name:14s} int4 absmax: {rel_mse(quantize(wt, 4, granularity='group', group_size=64), wt):.4f}   NF4: {rel_mse(quantize_nf4(wt), wt):.4f}")
qm = quantize_model(model, quantize_nf4); print("perplexity NF4:", round(perplexity(qm, tok, eval_text, max_windows=MAXW), 2)); del qm

## 3. Выбросы в активациях

Соберите хуком входы линейных слоёв `down_proj` (или аналогичных) и посчитайте максимум \(|x|\) по каждому входному каналу. Найдите каналы-выбросы (максимум в 20 и более раз выше медианы по каналам). Сравните ошибку выхода слоя при:
1. int8 per-tensor квантизации активаций;
2. int8 per-token квантизации;
3. разложении LLM.int8(): каналы-выбросы остаются в FP, остальные — int8 per-token.

In [ ]:
acts = {}
def save_input(name):
    def hook(mod, inp, out): acts.setdefault(name, []).append(inp[0].detach().reshape(-1, inp[0].size(-1)))
    return hook
hooks = [m.register_forward_hook(save_input(n)) for n, m in model.named_modules() if n.endswith("down_proj")]
calib_ids = tok(eval_text, return_tensors="pt").input_ids[0]
calib_ids = calib_ids[: len(calib_ids) // 512 * 512].view(-1, 512)[:8]      # до 8 окон по 512 токенов (около 4 тыс. токенов)
with torch.no_grad():
    for w in calib_ids: model(w[None].to(DEVICE))
for h in hooks: h.remove()
acts = {k: torch.cat(v) for k, v in acts.items()}

# TODO: ваш код (статистика выбросов по слоям и сравнение трёх схем на слое с наибольшими выбросами)

## 4. Упрощённый GPTQ для одного слоя

Возьмите слой `down_proj` из п. 3 и калибровочные активации \(X\): первые 3000 токенов — калибровка, остальные — отложенная выборка. Число калибровочных токенов должно заметно превышать \(d_{in}\) (1536 у `down_proj` SmolLM2-135M), иначе \(H\) вырождена и GPTQ переобучается на калибровку. Реализуйте GPTQ без ленивых блочных обновлений:
1. \(H = 2X^\top X + \lambda I\), \(\lambda = 0.01\cdot\mathrm{mean}(\mathrm{diag}\,H)\);
2. \(U = \mathrm{chol}(H^{-1})^\top\) (верхнетреугольная), так что \([H^{-1}]_{jj}\) и строки обновления берутся из \(U\);
3. для \(j = 1..d_{in}\): квантовать столбец \(w_{:,j}\) (int4, per-channel масштаб, заранее посчитанный по исходной матрице), \(\delta = (w_{:,j} - \hat w_{:,j})/U_{jj}\), \(W_{:,j+1:} \mathrel{-}= \delta\,U_{j,j+1:}\).

Сравните \(\|XW^\top - X\hat W^\top\|^2/\|XW^\top\|^2\) для RTN и GPTQ на калибровочной и на отложенной выборке активаций.

In [ ]:
def gptq_layer(W, X, bits=4, damp=0.01):
    W = W.clone().float(); d_in = W.size(1)
    qmax = 2 ** (bits - 1) - 1; scale = W.abs().amax(1, keepdim=True).clamp(min=1e-8) / qmax
    q = lambda col: (col / scale[:, 0]).round().clamp(-qmax, qmax) * scale[:, 0]
    # TODO: ваш код
    raise NotImplementedError

lin = dict(model.named_modules())[worst]; Wl = lin.weight.data.float()
X_all = acts[worst].float(); X_cal, X_hold = X_all[:3000], X_all[3000:]
t0 = time.time(); W_gptq = gptq_layer(Wl, X_cal); print(f"GPTQ: {time.time() - t0:.1f} c")
W_rtn = quantize(Wl, 4, granularity="per_channel")
for name, Xe in [("калибровка", X_cal), ("отложенная", X_hold)]:
    ref = Xe @ Wl.T
    print(f"{name:11s} RTN: {rel_mse(Xe @ W_rtn.T, ref):.4e}   GPTQ: {rel_mse(Xe @ W_gptq.T, ref):.4e}")

## 5. Speculative sampling

### 5а. Проверка на игрушечных распределениях

Реализуйте один шаг speculative sampling для заданных распределений \(p\) (цель) и \(q\) (черновик) над 10 символами. Повторите 100 000 раз и проверьте критерием хи-квадрат, что распределение выхода совпадает с \(p\). Посчитайте долю принятых токенов и сравните с теоретической \(\sum_x \min(p(x), q(x))\).

In [ ]:
from scipy.stats import chisquare

def chi2_pvalue(counts, probs):
    """χ²-тест согласия; ожидаемые частоты нормируются точно к сумме наблюдений (иначе scipy ругается на округление)."""
    counts = np.asarray(counts, dtype=float); e = np.asarray(probs, dtype=float); e = e / e.sum() * counts.sum()
    return chisquare(counts, e).pvalue

def spec_step(p, q, gen):
    """Один шаг: возвращает (токен, принят ли черновик)."""
    # TODO: ваш код
    raise NotImplementedError

gen = torch.Generator().manual_seed(0)
p = torch.distributions.Dirichlet(torch.ones(10)).sample(); q = torch.distributions.Dirichlet(torch.ones(10)).sample()
N = 100_000 if not FAST_DEV_RUN else 30_000
res = [spec_step(p, q, gen) for _ in range(N)]
counts = np.bincount([r[0] for r in res], minlength=10)
print("χ²-тест против p: p-value =", round(chi2_pvalue(counts, p.numpy()), 3))
print("χ²-тест против q: p-value =", chi2_pvalue(counts, q.numpy()))
print(f"доля принятых: {np.mean([r[1] for r in res]):.4f}, теория: {torch.minimum(p, q).sum().item():.4f}")

### 5б. Speculative decoding для LLM

Draft — меньшая модель того же семейства (общий токенизатор). Алгоритм для длины черновика \(\gamma\):
1. draft авторегрессионно генерирует \(\gamma\) токенов, сохраняя \(q_i\);
2. target одним прямым проходом вычисляет \(p_i\) для всех \(\gamma + 1\) позиций;
3. токены принимаются слева направо по правилу из 5а; при первом отказе генерируется исправленный токен и шаг заканчивается; если все приняты — добавляется токен из \(p_{\gamma+1}\).

Для простоты KV-cache не используется: каждый вызов модели — полный проход по префиксу. Измерьте число вызовов target на сгенерированный токен, долю принятых токенов и wall-clock время в сравнении с обычной генерацией target-моделью при \(\gamma \in \{2, 4, 6\}\).

In [ ]:
target, _ = load_lm(TARGET); draft = model.to(torch.float16 if DEVICE == "cuda" else torch.float32)

@torch.no_grad()
def probs_of(m, ids): return m(ids).logits[0].float().softmax(-1)

@torch.no_grad()
def speculative_generate(ids, n_new, gamma=4, gen=None):
    stats = {"target_calls": 0, "draft_calls": 0, "accepted": 0, "proposed": 0}; start = ids.size(1)
    while ids.size(1) - start < n_new:
        # TODO: ваш код
        raise NotImplementedError
    return ids[:, :start + n_new], stats

@torch.no_grad()
def plain_generate(m, ids, n_new, gen=None):
    for _ in range(n_new): ids = torch.cat([ids, torch.multinomial(probs_of(m, ids)[-1], 1, generator=gen)[None]], 1)
    return ids

prompt = tok("The history of the city begins", return_tensors="pt").input_ids.to(DEVICE)
N_NEW = 40 if FAST_DEV_RUN else 100
g = torch.Generator(device=DEVICE).manual_seed(0) if DEVICE == "cuda" else torch.Generator().manual_seed(0)
t0 = time.time(); plain_generate(target, prompt, N_NEW, g); t_plain = time.time() - t0
rows = [{"метод": "target, обычная генерация", "время, c": t_plain, "вызовов target на токен": 1.0, "доля принятых": float("nan")}]
for gamma in (2, 4, 6):
    t0 = time.time(); out, st = speculative_generate(prompt, N_NEW, gamma, g); dt = time.time() - t0
    rows.append({"метод": f"speculative, γ={gamma}", "время, c": dt, "вызовов target на токен": st["target_calls"] / N_NEW,
                 "доля принятых": st["accepted"] / st["proposed"]})
display(pd.DataFrame(rows).round(3)); print(tok.decode(out[0]))

### 5в. Проверка распределения на реальных моделях

Для фиксированного промпта сгенерируйте первый токен 150 раз (в полном режиме 2000) обычным сэмплированием из target и speculative-методом (\(\gamma = 1\)). Сравните частоты с \(p\) критерием хи-квадрат, объединив токены вне top-10 в одну категорию.

In [ ]:
# TODO: ваш код (выборки первого токена, χ²-тест против точного распределения p)

## Вопросы

1. Почему per-channel квантизация весов почти не теряет качества по сравнению с per-tensor, а для активаций нужна per-token и обработка выбросов?
2. Для какого распределения весов NF4 оптимальна и в каком смысле? Почему на равномерном распределении она уступает int4?
3. Выведите правило обновления GPTQ из задачи OBS: минимизировать прирост ошибки при фиксации одного веса.
4. Докажите, что speculative sampling сохраняет распределение \(p\). Выведите ожидаемое число токенов на один вызов target при вероятности принятия \(\alpha\) и длине черновика \(\gamma\).
5. Почему без KV-cache ускорение по wall-clock меньше, чем по числу вызовов target? Когда speculative decoding не даёт ускорения?